In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import AutoTokenizer
from itertools import islice

# Stream OpenWebText without downloading the full dataset
dataset = load_dataset(
    "Skylion007/openwebtext",
    split="train",
    streaming=True
)

# Take 1,000 documents for EDA
sample_size = 1000
sample = list(islice(dataset, sample_size))

# Convert the sample into a DataFrame
df = pd.DataFrame(sample)

print("OpenWebText sample loaded successfully.")
print("Sample size:", len(df))

In [ ]:
# Inspect the structure of the sampled dataset

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)

In [ ]:
# Preview the first OpenWebText document
print(df["text"].iloc[0][:300])

In [ ]:
# Check data quality of the OpenWebText sample

empty_documents = (df["text"].str.strip() == "").sum()
duplicate_documents = df["text"].duplicated().sum()

print("Empty documents:", empty_documents)
print("Duplicate documents:", duplicate_documents)

In [ ]:
# Calculate the number of words in each document
df["word_count"] = df["text"].str.split().str.len()

# Key document length statistics
print("Mean word count:", round(df["word_count"].mean(), 2))
print("Median word count:", df["word_count"].median())
print("Minimum word count:", df["word_count"].min())
print("Maximum word count:", df["word_count"].max())

In [ ]:
# Visualize the distribution of document lengths

plt.figure(figsize=(10, 5))

plt.hist(df["word_count"], bins=50, edgecolor="black")

plt.xlabel("Number of Words")
plt.ylabel("Number of Documents")
plt.title("Distribution of Document Lengths in OpenWebText Sample")

plt.savefig("../results/figures/eda_word_count_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Load the GPT-2 tokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

print("GPT-2 tokenizer loaded successfully.")
print("Vocabulary size:", tokenizer.vocab_size)

In [ ]:
# Check the maximum sequence length supported by GPT-2
print("GPT-2 maximum context length:", tokenizer.model_max_length, "tokens")

In [ ]:
# Take a short text segment from the first document
example_text = df["text"].iloc[0][:100]

# Convert the text into GPT-2 tokens
token_ids = tokenizer.encode(example_text, add_special_tokens=False)
tokens = tokenizer.convert_ids_to_tokens(token_ids)

print("Original text:")
print(example_text)

print("\nGPT-2 tokens:")
print(tokens)

print("\nNumber of tokens:", len(tokens))

In [ ]:
# Calculate GPT-2 token count for each document
df["token_count"] = df["text"].apply(
    lambda text: len(tokenizer.encode(text, add_special_tokens=False))
)

print("Mean token count:", round(df["token_count"].mean(), 2))
print("Median token count:", df["token_count"].median())
print("Minimum token count:", df["token_count"].min())
print("Maximum token count:", df["token_count"].max())

In [ ]:
# Visualize GPT-2 token length distribution

plt.figure(figsize=(10, 5))

plt.hist(df["token_count"], bins=50, edgecolor="black")

plt.xlabel("Number of GPT-2 Tokens")
plt.ylabel("Number of Documents")
plt.title("Distribution of GPT-2 Token Lengths in OpenWebText Sample")

plt.savefig("../results/figures/eda_token_count_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# GPT-2 Small maximum context length
gpt2_max_length = tokenizer.model_max_length

# Count documents within and beyond GPT-2's context limit
within_limit = (df["token_count"] <= gpt2_max_length).sum()
beyond_limit = (df["token_count"] > gpt2_max_length).sum()

print(f"Documents within 1024 tokens: {within_limit}")
print(f"Documents exceeding 1024 tokens: {beyond_limit}")

In [ ]:
# Visualize document token lengths with GPT-2's context limit

plt.figure(figsize=(10, 5))

plt.hist(df["token_count"], bins=50, edgecolor="black")

# GPT-2 maximum context boundary
plt.axvline(
    x=1024,
    linestyle="--",
    linewidth=2,
    label="GPT-2 Context Limit (1024 tokens)"
)

plt.xlabel("Number of GPT-2 Tokens")
plt.ylabel("Number of Documents")
plt.title("OpenWebText Token Lengths vs GPT-2 Context Limit")
plt.legend()

plt.savefig("../results/figures/eda_token_vs_context_limit.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Save EDA summary tables to results/tables
import os

tables_dir = "../results/tables"
os.makedirs(tables_dir, exist_ok=True)

# Data quality summary
quality_df = pd.DataFrame([{
    "empty_documents": int(empty_documents),
    "duplicate_documents": int(duplicate_documents),
    "sample_size": len(df),
}])
quality_df.to_csv(f"{tables_dir}/eda_data_quality.csv", index=False)

# Word count summary stats
df["word_count"].describe().to_csv(f"{tables_dir}/eda_word_count_stats.csv")

# Token count summary stats
df["token_count"].describe().to_csv(f"{tables_dir}/eda_token_count_stats.csv")

# Context window compliance
compliance_df = pd.DataFrame([{
    "within_1024_tokens": int(within_limit),
    "beyond_1024_tokens": int(beyond_limit),
    "pct_beyond": round(100 * beyond_limit / len(df), 2),
}])
compliance_df.to_csv(f"{tables_dir}/eda_context_window_compliance.csv", index=False)

print("Tables saved to", tables_dir)